# 混合正規分布モデル（GMM）による異常検知

前節で解説した多変数ホテリング理論やマハラノビス・タグチ法は、正常データが多次元正規分布に従うことを前提にしています。

一方で、可視化により正常データの分布が正規分布から⼤きく逸脱していると判断できる場合、単一の正規分布ではなく、より柔軟な分布モデルを用いる必要があります。

可視化の結果、正常分布が複数のピークを持つことが確認できた場合、以下の図のように複数の正規分布を重ね合わせた**GMM**（Gaussian mixture model：混合正規分布モデル）が有力なモデル候補となります。

<img src="images/fig6_11.png" width="30%">

ここではGMMによる異常検知を、Pythonを用いて以下の手順で実装する方法を解説します。

- A. モデルの学習
- B. 推論

今回は対象データとして複数ピークを持つデータを模擬するために、2クラスタのGMMに基づき生成したサンプルデータを使用し、このクラスタをGMMの学習でフィッティングできるかを検証してみます。

## A. モデルの学習

学習フェーズでは、EMアルゴリズムによるGMMパラメータの推定、および異常度の分位点に基づくしきい値算出を行います。Pythonでは、scikit-learnの[sklearn.mixture.GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html)クラスを用いると、GMMのEMアルゴリズムによる学習を簡単に実装できます。

In [ ]:
# コード6.5 GMMとEMアルゴリズムよる異常検知の実装例（学習）
import numpy as np
from sklearn.mixture import GaussianMixture

###### 学習データの読み込みと前処理######
rng = np.random.default_rng(seed=42) # 乱数シードを指定（結果の再現性を担保）
# 学習用サンプルとして2パターンの2次元正規分布から300個ずつデータ生成
MU_1 = [0, 0]
SIGMA_1 = [[1, 0],
           [0, 1]]
X_train_1 = rng.multivariate_normal(MU_1, SIGMA_1, 300)
MU_2 = [4, 0]
SIGMA_2 = [[2, 1],
           [1, 1]]
X_train_2 = rng.multivariate_normal(MU_2, SIGMA_2, 300)
X_train = np.concatenate([X_train_1, X_train_2])

###### 学習ステップ1. 正常のモデルを作成する######
gmm = GaussianMixture(n_components=2, random_state=42) # GMMモデルを作成
gmm.fit(X_train) # モデルを学習
mu_1 = gmm.means_[0] # 推定された1個目の正規分布の平均ベクトルμ1
Sigma_1 = gmm.covariances_[0] # 推定された1個目の正規分布の分散共分散行列Σ1
weight_1 = gmm.weights_[0] # 推定された1個目の正規分布の重みπ1
mu_2 = gmm.means_[1] # 推定された2個目の正規分布の平均ベクトルμ2
Sigma_2 = gmm.covariances_[1] # 推定された2個目の正規分布の分散共分散行列Σ2
weight_2 = gmm.weights_[1] # 推定された2個目の正規分布の重みπ2

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
x_train_anom_score = -gmm.score_samples(X_train)
# 異常度の分位点からしきい値を算出
a_th = np.quantile(x_train_anom_score, 1-TARGET_FP_RATE)

###### 学習で求めたパラメータをすべて表示######
print(f'mu_1={mu_1}')
print(f'Sigma_1={Sigma_1}')
print(f'weight_1={weight_1}')
print(f'mu_2={mu_2}')
print(f'Sigma_2={Sigma_2}')
print(f'weight_2={weight_2}')
print(f'a_th={a_th}')

求めたパラメータ`mu_1`、`mu_2`（各正規分布の標本平均ベクトル$\hat{\mu}_1,\hat{\mu}_2$）、`sigma_1`、`sigma_2`（各正規分布の標本分散共分散行$\hat{\Sigma}_1,\hat{\Sigma}_2$）、`weight_1`、`weight_2`（各正規分布の重み$\hat{\pi}_1,\hat{\pi}_2$）、`a_th`（異常度のしきい値$a_{th}$）は設定ファイルなどに保存して、推論フェーズで再利用します（ここでは簡単のため、得られたパラメータを推論時に直接コードに記述することとします）。

推定されたGMMの確率密度関数と学習データを重ねてプロットしてみます。

In [ ]:
# 学習データとモデルの確率密度関数を可視化
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats

###### GMMの確率密度関数を描画 ######
# (x1,x2)格子点を作成（'temp2'をx1としていることに注意）
x1_min, x1_max = np.min(X_train[:, 0]), np.max(X_train[:, 0])
x2_min, x2_max = np.min(X_train[:, 1]), np.max(X_train[:, 1])
x1_grid = np.linspace(x1_min, x1_max, num=200)
x2_grid = np.linspace(x2_min, x2_max, num=200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
X_grid = np.c_[X1.ravel(), X2.ravel()]
# 確率密度関数
X_grid_p1 = stats.multivariate_normal.pdf(  # 1個目の正規分布の確率密度関数
    X_grid, mean=mu_1, cov=Sigma_1)
X_grid_p2 = stats.multivariate_normal.pdf(  # 2個目の正規分布の確率密度関数
    X_grid, mean=mu_2, cov=Sigma_2)
X_grid_pd = weight_1*X_grid_p1 + weight_2*X_grid_p2  # 両者のGMMの確率密度
# 確率密度をプロット
X_grid_pivot = X_grid_pd.reshape(X1.shape)  # ピボット化
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))
ax.contourf(X1, X2, X_grid_pivot, levels=10, cmap=cm.gray, alpha=0.5)

###### 学習データを散布図で描画 ######
sns.scatterplot(x=X_train_1[:, 0], y=X_train_1[:, 1],
                c='#333333', ax=ax, s=18, marker="o", label="k=1")
sns.scatterplot(x=X_train_2[:, 0], y=X_train_2[:, 1],
                c='#333333', ax=ax, s=24, marker="^", label="k=2")
ax.set_xlabel('x1')
ax.set_ylabel('x2')
# 凡例を追加
ax.legend()
# グラフを表示
plt.show()

データに合わせてうまく確率密度関数が推定できていることがわかります。

## B. 推論

学習フェーズで求めたパラメータとしきい値を用いて、推論データに対する異常度の算出と異常判定を行います（今回使用する推論データは簡単のため、学習データと同様のGMMから生成された模擬データを正常データ、それとは異なる正規分布から生成された模擬データを異常データとして用いることとします）。

In [ ]:
# コード6.8 GMM とEM アルゴリズムよる異常検知の実装例（推論）
import pandas as pd
from scipy import stats

###### 学習したパラメータをここに記載
MU_1 = [-0.02919588, -0.02712137] # 平均ベクトルμ1
SIGMA_1 = [[0.96375596, 0.17096911],
           [0.17096911, 0.88357016]] # 分散共分散行列Σ1
WEIGHT_1 = 0.5038811475819546 # 重みπ1
MU_2 = [4.08312105, 0.02069242] # 平均ベクトルμ2
SIGMA_2 = [[1.91071117, 0.85054683],
           [0.85054683, 0.8488193]] # 分散共分散行列Σ2
WEIGHT_2 = 0.49611885241804554 # 重みπ2
A_TH=7.711945069689567 # 異常度のしきい値

###### 推論データの読み込みと前処理######
# 推論用の正常データとして二つの正規分布からデータを200個ずつ生成
rng = np.random.default_rng(seed=42)
X_inference_norm_1 = rng.multivariate_normal([0, 0], [[1, 0], [0, 1]], 100)
X_inference_norm_2 = rng.multivariate_normal([4, 0], [[2, 1], [1, 1]], 100)
X_inference_norm = np.concatenate([X_inference_norm_1, X_inference_norm_2])
df_inference_norm = pd.DataFrame(X_inference_norm, columns=["x1", "x2"])
df_inference_norm['label'] = 'normal'
# 推論用の異常データとして平均[3, 2]、2の正規分布からデータを20個生成
X_inference_anom = rng.multivariate_normal([3, 2], [[1, 0], [0, 1]], 20)
df_inference_anom = pd.DataFrame(X_inference_anom, columns=["x1", "x2"])
df_inference_anom['label'] = 'anomaly'
# 推論用の正常データと異常データを合体
df_inference = pd.concat([df_inference_norm, df_inference_anom], axis=0)
df_inference = df_inference.reset_index(drop=True)
X_inference = df_inference[["x1", "x2"]].to_numpy()

###### 推論を実行######
# 異常度を算出
X_inference_p1 = stats.multivariate_normal.pdf( # 1個目の正規分布の確率密度関数
    X_inference, mean=MU_1, cov=SIGMA_1)
X_inference_p2 = stats.multivariate_normal.pdf( # 2個目の正規分布の確率密度関数
    X_inference, mean=MU_2, cov=SIGMA_2)
# 両者のGMMの確率密度
X_inference_pd = WEIGHT_1*X_inference_p1 + WEIGHT_2*X_inference_p2
anomaly_scores = -np.log(X_inference_pd) # 異常度を求める
# しきい値により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, 'anomaly', 'normal')
# 推論結果を表示
print(pred)

推論結果の決定境界（異常と正常の判定の境界）を可視化してみます。

In [ ]:
# 推論結果の決定境界を可視化
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# 描画用のFigureとAxesを生成
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

###### 正常と異常の範囲を色分け ######
# (x1,x2)格子点を作成（'temp2'をx1としていることに注意）
x1_grid = np.linspace(df_inference['x1'].min()-2,
                      df_inference['x1'].max()+2, 200)
x2_grid = np.linspace(df_inference['x2'].min()-1,
                      df_inference['x2'].max()+1, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
X_grid = np.c_[X1.ravel(), X2.ravel()]
# 異常度を算出（式4.114に従う）
X_grid_p1 = stats.multivariate_normal.pdf(  # 1個目の正規分布の確率密度関数
    X_grid, mean=MU_1, cov=SIGMA_1)
X_grid_p2 = stats.multivariate_normal.pdf(  # 2個目の正規分布の確率密度関数
    X_grid, mean=MU_2, cov=SIGMA_2)
X_grid_pd = WEIGHT_1*X_grid_p1 + WEIGHT_2*X_grid_p2  # 両者のGMMの確率密度
anomaly_scores_grid = -np.log(X_grid_pd)  # 異常度を求める
# しきい値判定
pred_grid = np.where(anomaly_scores_grid > A_TH, 0, 1)
# 正常と異常の境界をプロット
pred_pivot = pred_grid.reshape(X1.shape)
ax.contourf(X1, X2, pred_pivot,
            cmap=cm.gray, alpha=0.5)

###### 各データを散布図としてプロット ######
sns.scatterplot(data=df_inference, x='x1', y='x2',
    hue='label', palette=['#999999', '#111111'],
    ax=ax
)
# 凡例を追加
ax.legend()
# グラフを表示
plt.show()

背景の塗りつぶし色がしきい値判定の結果を表しており、暗い部分が異常判定を表します。また散布図が推論に使用したデータ点を表しており、マーカー色（凡例）が正解ラベルを示します。

GMMの確率密度関数に基づき決定境界が引かれていることが分かります。